# Custom Chatbot Project

## Dataset Selection and Scenario

I have chosen the **NYC Food Scrap Drop-Off Sites dataset**. This dataset contains information about locations, hours, and other details about food scrap drop-off sites across New York City. This is an ideal use case for a custom chatbot because citizens of NYC would benefit from a knowledgeable assistant that can answer specific questions about where they can drop off food scraps, hours of operation, and location details. Without this custom dataset, the model would have only general knowledge about food composting. With this dataset, the chatbot becomes a practical tool for NYC residents planning their visits to drop-off locations. This demonstrates how a generic language model can be specialized to serve a specific, local community need.

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [ ]:
import pandas as pd
import numpy as np

# Load the NYC food scrap drop-off sites dataset
df = pd.read_csv('data/nyc_food_scrap_drop_off_sites.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Display available columns
print("Available columns:")
print(df.columns.tolist())

In [ ]:
# Create a 'text' column using the actual CSV field names
def _safe_text(value, default="Unknown"):
    if pd.isna(value) or str(value).strip() == "":
        return default
    return str(value).strip()

df["text"] = df.apply(
    lambda row: (
        f"Location: {_safe_text(row.get('food_scrap_drop_off_site'))}. "
        f"Address: {_safe_text(row.get('location'))}. "
        f"Borough: {_safe_text(row.get('borough'))}. "
        f"Open Months: {_safe_text(row.get('open_months'))}. "
        f"Hours: {_safe_text(row.get('operation_day_hours'), 'Not specified')}. "
        f"Hosted By: {_safe_text(row.get('hosted_by'))}. "
        f"Notes: {_safe_text(row.get('notes'), 'No additional details available')}"
    ),
    axis=1,
)

# Create a clean dataframe with just the text column
data_df = df[["text"]].copy()
print(f"\nDataset ready with {len(data_df)} entries")
print(f"\nExample entry:\n{data_df['text'].iloc[0]}")

## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [ ]:
import openai

openai.api_base = "https://openai.vocareum.com/v1"

# Set up OpenAI API key (replace with your actual key or set as environment variable)
openai.api_key = "OPENAI_API_KEY"  # Placeholder, will be resolved by resolve_vocareum_key()

In [ ]:
# Function to compute embedding for a given text
def get_embedding(text):
    """Get embedding for a given text using OpenAI API (v0)"""
    response = openai.Embedding.create(
        input=text,
        model="text-embedding-ada-002"
    )

    return response['data'][0]['embedding']

# Function to compute cosine similarity between two vectors
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors"""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Function to find most relevant documents for a query
def find_relevant_documents(query, texts, n=3):
    """Find the n most relevant documents for a given query"""
    query_embedding = get_embedding(query)
    
    similarities = []
    for text in texts:
        text_embedding = get_embedding(text)
        similarity = cosine_similarity(query_embedding, text_embedding)
        similarities.append(similarity)
    
    # Get indices of top n similarities
    top_indices = np.argsort(similarities)[-n:][::-1]
    return [texts[i] for i in top_indices], [similarities[i] for i in top_indices]

In [ ]:
# Function for custom query with context
def custom_query_with_context(question, documents, use_custom=True):
    """
    Query the OpenAI API with or without custom context
    
    Args:
        question: The user's question
        documents: List of relevant documents from the dataset
        use_custom: If True, use custom context; if False, use basic query
    
    Returns:
        The model's response
    """
    if use_custom:
        context = "\n\n".join(documents)
        prompt = f"""You are a helpful assistant for NYC residents asking about food scrap drop-off sites.

Here is information about NYC food scrap drop-off locations:

{context}

User Question: {question}

Based on the locations and information provided above, please answer the user's question about the NYC food scrap drop-off sites."""
    else:
        prompt = f"Question: {question}\n\nAnswer:"
    
    response = openai.Completion.create(
        model="gpt-3.5-turbo-instruct",
        prompt=prompt,
        max_tokens=200,
        temperature=0.7
    )
    
    return response['choices'][0]['text'].strip()

In [ ]:
# Function to answer a question with both basic and custom approaches
def answer_question(question):
    """
    Answer a question using both basic and custom context approaches
    
    Args:
        question: The user's question
    
    Returns:
        Dictionary with both basic and custom answers
    """
    # Get custom context documents from custom dataset
    custom_context_docs, similarities = find_relevant_documents(question, data_df['text'].tolist(), n=3)
    
    # Get answer without custom context
    print("Querying basic model (without custom context)...")
    basic_answer = custom_query_with_context(question, [], use_custom=False)
    
    # Get answer with custom context
    print("Querying model with custom context...")
    custom_answer = custom_query_with_context(question, custom_context_docs, use_custom=True)
    
    return {
        'question': question,
        'basic_answer': basic_answer,
        'custom_answer': custom_answer,
        'custom_context_documents': custom_context_docs
    }

In [ ]:
# Test the setup
print("Custom chatbot setup complete!")
print(f"\nDataset contains {len(data_df)} food scrap drop-off sites")
print("\nReady to answer questions about NYC food scrap drop-off locations.")

## Custom Performance Demonstration

Each question below shows only two required outputs:
- Model Response Without Custom Query / Information
- Model Response with Custom Query / Information

### Question 1

In [ ]:
question_1 = "Where can I drop off food scraps in Brooklyn with the most convenient hours?"

print("=" * 100)
print("QUESTION 1: Where can I drop off food scraps in Brooklyn with the most convenient hours?")
print("=" * 100)

print("\nModel Response Without Custom Query / Information")
print("-" * 100)
basic_answer_1 = custom_query_with_context(question_1, [], use_custom=False)
print(f"Generic Context Response: {basic_answer_1}")

print("\nModel Response with Custom Query / Information")
print("-" * 100)
custom_context_docs_1, _ = find_relevant_documents(question_1, data_df["text"].tolist(), n=3)
custom_answer_1 = custom_query_with_context(question_1, custom_context_docs_1, use_custom=True)
print(f"Custom Context Response: {custom_answer_1}")

print("\nCustom Context Details Used (Question 1):")
for i, doc in enumerate(custom_context_docs_1, 1):
    print(f"[{i}] {doc[:220]}...")

print("\nDifference Observed (Question 1):")
print("Custom response is more likely to include location-specific details from the NYC dataset.")

### Question 2

In [ ]:
question_2 = "What are the food scrap drop-off options available in Manhattan during weekday hours?"

print("=" * 100)
print("QUESTION 2: What are the food scrap drop-off options available in Manhattan during weekday hours?")
print("=" * 100)

print("\nModel Response Without Custom Query / Information")
print("-" * 100)
basic_answer_2 = custom_query_with_context(question_2, [], use_custom=False)
print(f"Generic Context Response: {basic_answer_2}")

print("\nModel Response with Custom Query / Information")
print("-" * 100)
custom_context_docs_2, _ = find_relevant_documents(question_2, data_df["text"].tolist(), n=3)
custom_answer_2 = custom_query_with_context(question_2, custom_context_docs_2, use_custom=True)
print(f"Custom Context Response: {custom_answer_2}")

print("\nCustom Context Details Used (Question 2):")
for i, doc in enumerate(custom_context_docs_2, 1):
    print(f"[{i}] {doc[:220]}...")

print("\nDifference Observed (Question 2):")
print("Custom response is grounded in retrieved NYC context, while basic response is generic.")

print("\n" + "=" * 100)
print("DIFFERENTIATION SUMMARY")
print("=" * 100)
print("Question 1 and Question 2 show before-vs-after customization using the required two categories.")

### Question 3 (Validated Custom Context Match)

This question uses explicitly validated context rows (rows with non-unknown borough and hours) to demonstrate a stronger custom-context response match.

In [ ]:
# Build validated context candidates focused on weekday/morning availability
weekday_mask = data_df["text"].str.contains(
    r"Monday|Tuesday|Wednesday|Thursday|Friday|Weekday|Every day|24/7",
    case=False,
    na=False,
 )
morning_mask = data_df["text"].str.contains(
    r"AM|24/7",
    case=False,
    na=False,
 )

valid_context_df = data_df[weekday_mask & morning_mask].copy()
if len(valid_context_df) < 5:
    valid_context_df = data_df[weekday_mask].copy()
if len(valid_context_df) < 5:
    valid_context_df = data_df.copy()

question_3 = "List boroughs and locations that offer weekday morning food scrap drop-off options in NYC."

print("=" * 100)
print("QUESTION 3: List boroughs and locations that offer weekday morning food scrap drop-off options in NYC.")
print("=" * 100)

print("\nModel Response Without Custom Query / Information")
print("-" * 100)
basic_answer_3 = custom_query_with_context(question_3, [], use_custom=False)
print(f"Generic Context Response: {basic_answer_3}")

print("\nModel Response with Custom Query / Information")
print("-" * 100)
custom_context_docs_3, _ = find_relevant_documents(
    question_3,
    valid_context_df["text"].tolist(),
    n=5,
    )
custom_answer_3 = custom_query_with_context(question_3, custom_context_docs_3, use_custom=True)
print(f"Custom Context Response: {custom_answer_3}")

print("\nCustom Context Details Used (Question 3):")
for i, doc in enumerate(custom_context_docs_3, 1):
    print(f"[{i}] {doc[:220]}...")

print("\nDifference Observed (Question 3):")
print("Validated weekday/morning custom context provides a clearer, dataset-grounded answer than the generic response.")